# OCR Farmer Feedback to Text
Convert the `farmer_feedback_observations_lots_additional.jpg` image into a TXT file stored under `00_docs`.

In [2]:
from pathlib import Path

import easyocr

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "00_docs").exists() else cwd.parent
image_path = repo_root / "00_docs" / "farmer_feedback_observations_lots_additional.jpg"
text_path = repo_root / "00_docs" / "farmer_feedback_observations_lots_additional_ocr.txt"

if not image_path.exists():
    raise FileNotFoundError(f"Image not found: {image_path}")

reader = easyocr.Reader(["en"], gpu=False)
results = reader.readtext(str(image_path), detail=0, paragraph=True)
ocr_text = "\n".join(line.strip() for line in results if line.strip())

text_path.write_text(ocr_text.strip() + "\n", encoding="utf-8")
print(f"Wrote OCR text to {text_path}")

Using CPU. Note: This module is much faster with a GPU.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

c:\Users\msario\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Wrote OCR text to C:\Users\msario\OneDrive - Microsoft\2026\00_All about Fabric\Fabric IQ Demo\FabricIQ-Manufacturing\00_docs\farmer_feedback_observations_lots_additional_ocr.txt


In [ ]:
"""
This code sample shows Prebuilt Read operations with the Azure AI Document Intelligence client library
using Microsoft Entra authentication (managed identity via DefaultAzureCredential).

Reference:
https://learn.microsoft.com/azure/ai-services/document-intelligence/quickstarts/get-started-sdks-rest-api?pivots=programming-language-python
"""

import os
import numpy as np

from azure.identity import DefaultAzureCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest

# Set this in your environment, for example:
# $env:DOCUMENT_INTELLIGENCE_ENDPOINT="https://<your-resource>.cognitiveservices.azure.com/"
endpoint = os.getenv("DOCUMENT_INTELLIGENCE_ENDPOINT", "")
if not endpoint:
    raise ValueError("Set DOCUMENT_INTELLIGENCE_ENDPOINT environment variable.")


def format_bounding_box(bounding_box):
    if not bounding_box:
        return "N/A"
    reshaped_bounding_box = np.array(bounding_box).reshape(-1, 2)
    return ", ".join(["[{}, {}]".format(x, y) for x, y in reshaped_bounding_box])


def analyze_read():
    # Sample document URL
    form_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-layout.pdf"

    # Managed identity / Entra auth via DefaultAzureCredential
    credential = DefaultAzureCredential()
    document_intelligence_client = DocumentIntelligenceClient(
        endpoint=endpoint,
        credential=credential,
    )

    poller = document_intelligence_client.begin_analyze_document(
        "prebuilt-read",
        AnalyzeDocumentRequest(url_source=form_url),
    )
    result = poller.result()

    print("Document contains content:", result.content)

    for style in result.styles:
        print(
            "Document contains {} content".format(
                "handwritten" if style.is_handwritten else "no handwritten"
            )
        )

    for page in result.pages:
        print("----Analyzing Read from page #{}----".format(page.page_number))
        print(
            "Page has width: {} and height: {}, measured with unit: {}".format(
                page.width,
                page.height,
                page.unit,
            )
        )

        for line_idx, line in enumerate(page.lines):
            print(
                "...Line # {} has text content '{}' within bounding box '{}'".format(
                    line_idx,
                    line.content,
                    format_bounding_box(line.polygon),
                )
            )

        for word in page.words:
            print(
                "...Word '{}' has a confidence of {}".format(
                    word.content,
                    word.confidence,
                )
            )

    print("----------------------------------------")


if __name__ == "__main__":
    analyze_read()

Run Cell 2 for local OCR output to `00_docs`, then run Cell 3 for Azure Document Intelligence using managed identity.

Prerequisites for Cell 3:
- `azure-ai-documentintelligence`
- `azure-identity`
- `numpy`
- `DOCUMENT_INTELLIGENCE_ENDPOINT` environment variable is set
- Your managed identity has access to the Document Intelligence resource